In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, vstack
from scipy.sparse.linalg import svds
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [6]:
books = pd.read_csv('Books.csv', sep=';', encoding='latin-1', on_bad_lines='skip', low_memory=False)
ratings = pd.read_csv('Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
users = pd.read_csv('Users.csv', sep=';', encoding='latin-1', on_bad_lines='skip')

# Стандартизуем имена колонок (зависит от версии датасета)
books.columns = [c.strip().replace('"','') for c in books.columns]
ratings.columns = [c.strip().replace('"','') for c in ratings.columns]

print("Books:", books.shape, "| Ratings:", ratings.shape, "| Users:", users.shape)
print(books.columns.tolist())
print(ratings.head())
print(books.head())
print(users.head())

Books: (271379, 5) | Ratings: (1149780, 3) | Users: (278859, 2)
['ISBN', 'Title', 'Author', 'Year', 'Publisher']
   User-ID        ISBN  Rating
0   276725  034545104X       0
1   276726  0155061224       5
2   276727  0446520802       0
3   276729  052165615X       3
4   276729  0521795028       6
         ISBN                                              Title  \
0  0195153448                                Classical Mythology   
1  0002005018                                       Clara Callan   
2  0060973129                               Decision in Normandy   
3  0374157065  Flu: The Story of the Great Influenza Pandemic...   
4  0393045218                             The Mummies of Urumchi   

                 Author  Year                Publisher  
0    Mark P. O. Morford  2002  Oxford University Press  
1  Richard Bruce Wright  2001    HarperFlamingo Canada  
2          Carlo D'Este  1991          HarperPerennial  
3      Gina Bari Kolata  1999     Farrar Straus Giroux  
4      

In [3]:
print("Распределение оценок:")
print(ratings['Book-Rating'].value_counts().sort_index())

ratings['Book-Rating'].hist(bins=11)
plt.title('Распределение оценок (0 = неявная)')
plt.xlabel('Rating'); plt.ylabel('Count'); plt.show()
# Только явные оценки
ratings = ratings[ratings['Book-Rating'] > 0].copy()

# Фильтрация sparsity
u_cnt = ratings['User-ID'].value_counts()
b_cnt = ratings['ISBN'].value_counts()
ratings = ratings[ratings['User-ID'].isin(u_cnt[u_cnt >= 15].index)]
ratings = ratings[ratings['ISBN'].isin(b_cnt[b_cnt >= 15].index)]
books   = books[books['ISBN'].isin(ratings['ISBN'].unique())]

print(f"После фильтрации: {ratings.shape[0]} оценок, "
      f"{ratings['User-ID'].nunique()} юзеров, {ratings['ISBN'].nunique()} книг")
print(f"Sparsity: {1 - ratings.shape[0] / (ratings['User-ID'].nunique() * ratings['ISBN'].nunique()):.4f}")

Распределение оценок:


KeyError: 'Book-Rating'